# Phase 2: Tokenization & Data Preparation

**Project:** Completing Chopin's Lost Waltz using AI  
**Goal:** Convert 19 Chopin waltz MIDI files into tokenized, augmented, chunked training data.

This notebook:
1. Downloads the pre-trained REMI+BPE tokenizer from the Maestro model
2. Splits waltzes into train/validation sets (16 train, 3 validation)
3. Chunks long waltzes into ≤1024-token segments
4. Augments training data via pitch transposition, velocity & duration offsets
5. Tokenizes the lost waltz separately (seed for generation)
6. Verifies everything and exports statistics

**Runs entirely on laptop — no GPU needed yet.**

## 2.0 — Setup

In [15]:
from pathlib import Path
from random import shuffle, seed
import json

from miditok import REMI
from miditok.data_augmentation import augment_dataset
from miditok.utils import split_files_for_training
from symusic import Score

# ──────────────────────────────────────────────
# CONFIGURE THESE PATHS
# ──────────────────────────────────────────────
RAW_DIR = Path("../data/raw").resolve()             # Your 20 MIDI files
CHUNKED_DIR = Path("../data/chunked").resolve()    # Output: chunked MIDI files
SEED_DIR = Path("../data/seed").resolve()           # Output: tokenized lost waltz
ANALYSIS_DIR = Path("../analysis").resolve()        # Output: stats and logs
# ──────────────────────────────────────────────

# Create output directories
for d in [CHUNKED_DIR, SEED_DIR, ANALYSIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# List all MIDI files
all_midi = sorted(RAW_DIR.glob("*.mid"))
print(f"Found {len(all_midi)} MIDI files in {RAW_DIR.resolve()}")
for f in all_midi:
    print(f"  {f.name}")

Found 20 MIDI files in /Users/jawadmallat/Desktop/Chopin_Waltz_Completion/data/raw
  New_Chopin_Waltz_.mid
  Waltz_10_op69no2-in-B-minor.mid
  Waltz_11_op70no1-in-Gb-major.mid
  Waltz_12_op70no2-in-F-minor.mid
  Waltz_13_op70no3-in-Db-major.mid
  Waltz_14_B56-in-E-minor.mid
  Waltz_15_B44-in-E-major.mid
  Waltz_16_B21-in-Ab-major.mid
  Waltz_17_B46-in-Eb-major.mid
  Waltz_18_B133-in-Eb-major.mid
  Waltz_19_B150-in-A-minor.mid
  Waltz_1_op18-in-Eb-major.mid
  Waltz_2_op34no1-in-Ab-major.mid
  Waltz_3_op34no2-in-A-minor.mid
  Waltz_4_op34no3-in-F-major.mid
  Waltz_5_op42-in-Ab-major.mid
  Waltz_6_op64no1-in-Db-major.mid
  Waltz_7_op64no2-in-C#-minor.mid
  Waltz_8_op64no3-in-Ab-major.mid
  Waltz_9_op69no1-in-Ab-major.mid


## 2.1 — Load the Pre-trained Tokenizer

We use the **exact tokenizer** from the pre-trained Maestro-REMI-bpe20k model.  
This is critical: the model's weights expect this specific vocabulary (20,000 BPE tokens).  
Using any other tokenizer would make the pre-trained weights meaningless.

The first time you run this, it will download the tokenizer config (~1 MB) from HuggingFace.

In [16]:
# Download the pre-trained tokenizer from the Maestro model
from huggingface_hub import hf_hub_download

tokenizer_path = hf_hub_download(
    repo_id="Natooz/Maestro-REMI-bpe20k",
    filename="tokenizer.json",
)

# Load from the downloaded file
tokenizer = REMI(params=tokenizer_path)

print(f"Tokenizer loaded! Vocabulary size: {len(tokenizer)}")

Tokenizer loaded! Vocabulary size: 20000


/Users/jawadmallat/Desktop/Chopin_Waltz_Completion/venv/lib/python3.12/site-packages/miditok/classes.py:886: UserWarning: Argument nb_tempos has been renamed num_tempos, you should consider to updateyour code with this new argument name.
  return cls(**input_dict, **kwargs)


In [17]:
for i in range(min(10, len(tokenizer))):
    print(f"  {i}: {tokenizer[i]}")

print(f"\n...")
print(f"\nSpecial tokens:")
for name in ["PAD_None", "BOS_None", "EOS_None"]:
    try:
        print(f"  {name} -> id {tokenizer[name]}")
    except:
        print(f"  {name} -> not found (checking alternatives...)")

# Save tokenizer locally for future use
tokenizer_save_path = Path("../models/tokenizer.json")
tokenizer_save_path.parent.mkdir(parents=True, exist_ok=True)
tokenizer.save(tokenizer_save_path)
print(f"\nTokenizer saved to {tokenizer_save_path}")

  0: PAD_None
  1: MASK_None
  2: BOS_None
  3: EOS_None
  4: SEP_None
  5: Bar_None
  6: Pitch_21
  7: Pitch_22
  8: Pitch_23
  9: Pitch_24

...

Special tokens:
  PAD_None -> id 0
  BOS_None -> id 2
  EOS_None -> id 3

Tokenizer saved to ../models/tokenizer.json


## 2.2 — Quick Tokenization Test on the Lost Waltz

Before processing the full corpus, let's verify the tokenizer works on one file.  
We tokenize the lost waltz and inspect what the tokens look like.

In [18]:
# Find the lost waltz
lost_waltz_path = RAW_DIR / "New_Chopin_Waltz_.mid"
assert lost_waltz_path.exists(), f"Lost waltz not found at {lost_waltz_path}"

# Tokenize it
lost_waltz_score = Score(str(lost_waltz_path))
lost_waltz_tokens = tokenizer(lost_waltz_score)

# Inspect
print(f"Lost waltz tokenization:")
print(f"  Number of track sequences: {len(lost_waltz_tokens)}")
for i, seq in enumerate(lost_waltz_tokens):
    print(f"  Track {i}: {len(seq.ids)} tokens")

# Show the first 50 token IDs (BPE tokens can't all be decoded by index)
first_seq = lost_waltz_tokens[0]
print(f"\nFirst 50 token IDs:")
print(first_seq.ids[:50])

# Decode back to token strings using the proper method
print(f"\nFirst 50 tokens (decoded):")
tokens_str = first_seq.tokens[:50] if hasattr(first_seq, 'tokens') and first_seq.tokens else None
if tokens_str:
    for j, t in enumerate(tokens_str):
        print(f"  [{j:3d}] id={first_seq.ids[j]:5d}  ->  {t}")
else:
    print("  (BPE tokens — decoded names not directly available, this is normal)")
    print("  The roundtrip test below will verify correctness instead.")

# Verify roundtrip: tokens -> MIDI -> listen
roundtrip_midi = tokenizer(lost_waltz_tokens)
roundtrip_path = SEED_DIR / "lost_waltz_roundtrip_test.mid"
roundtrip_midi.dump_midi(str(roundtrip_path))
print(f"\nRoundtrip MIDI saved to {roundtrip_path}")
print("Open this in MuseScore and compare with the original to verify quality.")

Lost waltz tokenization:
  Number of track sequences: 2
  Track 0: 656 tokens
  Track 1: 504 tokens

First 50 token IDs:
[175, 5071, 146, 1702, 150, 1234, 154, 1739, 158, 1234, 175, 1736, 134, 1414, 138, 1549, 142, 1414, 146, 50, 1799, 41, 1799, 154, 9381, 13897, 2288, 231, 134, 5626, 2371, 231, 142, 5626, 2369, 231, 150, 6193, 2192, 231, 158, 6193, 175, 50, 1799, 41, 1799, 2611, 231, 142]

First 50 tokens (decoded):
  [  0] id=  175  ->  Bar_None
  [  1] id= 5071  ->  Position_0
  [  2] id=  146  ->  Pitch_57
  [  3] id= 1702  ->  Velocity_79
  [  4] id=  150  ->  Duration_1.0.4
  [  5] id= 1234  ->  Position_16
  [  6] id=  154  ->  Pitch_52
  [  7] id= 1739  ->  Velocity_47
  [  8] id=  158  ->  Duration_0.4.8
  [  9] id= 1234  ->  Position_20
  [ 10] id=  175  ->  Pitch_60
  [ 11] id= 1736  ->  Velocity_47
  [ 12] id=  134  ->  Duration_0.4.8
  [ 13] id= 1414  ->  Position_24
  [ 14] id=  138  ->  Pitch_51
  [ 15] id= 1549  ->  Velocity_47
  [ 16] id=  142  ->  Duration_0.4.8
  [ 1

## 2.3 — Separate Corpus from Lost Waltz & Split Train/Validation

**Critical:** the lost waltz is NEVER in the training data.  
We split the 19 known waltzes: **16 for training, 3 for validation.**

Validation set strategy: we hold out one waltz from each key group  
(major flat keys, minor keys, major sharp keys) for balanced evaluation.

In [19]:
# Separate lost waltz from the 19 known waltzes
corpus_files = sorted([f for f in all_midi if "New_Chopin" not in f.name])
lost_waltz_file = [f for f in all_midi if "New_Chopin" in f.name]

print(f"Corpus: {len(corpus_files)} known waltzes")
print(f"Lost waltz: {len(lost_waltz_file)} file")
assert len(corpus_files) == 19, f"Expected 19 waltzes, got {len(corpus_files)}"

# ──────────────────────────────────────────────
# TRAIN/VALIDATION SPLIT
# Selected by Algorithm 3 (minimum training chunks lost):
# Remove the 3 smallest B-catalog major waltzes — least representative
# of Chopin's waltz style, and their removal loses the fewest training tokens.
# All minor-key waltzes and all concert waltzes stay in training.
# ──────────────────────────────────────────────

val_names = [
    "Waltz_18_B133-in-Eb-major.mid",    # 487 notes — shortest in corpus
    "Waltz_15_B44-in-E-major.mid",      # 1257 notes — early period
    "Waltz_16_B21-in-Ab-major.mid",     # 1335 notes — early period
]

val_files = [f for f in corpus_files if f.name in val_names]
train_files = [f for f in corpus_files if f.name not in val_names]

assert len(val_files) == 3, f"Expected 3 validation files, got {len(val_files)}"
assert len(train_files) == 16, f"Expected 16 training files, got {len(train_files)}"

print(f"\nTraining set ({len(train_files)} waltzes):")
for f in train_files:
    marker = " ★ A MINOR" if "A-minor" in f.name else ""
    print(f"  {f.name}{marker}")

print(f"\nValidation set ({len(val_files)} waltzes):")
for f in val_files:
    print(f"  {f.name}")

Corpus: 19 known waltzes
Lost waltz: 1 file

Training set (16 waltzes):
  Waltz_10_op69no2-in-B-minor.mid
  Waltz_11_op70no1-in-Gb-major.mid
  Waltz_12_op70no2-in-F-minor.mid
  Waltz_13_op70no3-in-Db-major.mid
  Waltz_14_B56-in-E-minor.mid
  Waltz_17_B46-in-Eb-major.mid
  Waltz_19_B150-in-A-minor.mid ★ A MINOR
  Waltz_1_op18-in-Eb-major.mid
  Waltz_2_op34no1-in-Ab-major.mid
  Waltz_3_op34no2-in-A-minor.mid ★ A MINOR
  Waltz_4_op34no3-in-F-major.mid
  Waltz_5_op42-in-Ab-major.mid
  Waltz_6_op64no1-in-Db-major.mid
  Waltz_7_op64no2-in-C#-minor.mid
  Waltz_8_op64no3-in-Ab-major.mid
  Waltz_9_op69no1-in-Ab-major.mid

Validation set (3 waltzes):
  Waltz_15_B44-in-E-major.mid
  Waltz_16_B21-in-Ab-major.mid
  Waltz_18_B133-in-Eb-major.mid


## 2.4 — Chunk & Augment: The Core Pipeline

For each subset (train and validation), we:
1. **Chunk** MIDIs into segments of ≤1024 tokens (with 2-bar overlap)
2. **Augment** the chunks with pitch transposition and velocity/duration offsets

The order matters: chunk first, then augment. This way augmentation creates  
varied copies of each chunk, maximizing training diversity.

**Augmentation parameters:**
- `pitch_offsets=[-12, 12]`: transpose down/up by one octave (12 semitones)
- `velocity_offsets=[-4, 4]`: slightly quieter/louder
- `duration_offsets=[-0.5, 0.5]`: slightly shorter/longer notes

⚠️ **We only augment the training set, NOT the validation set.**  
Validation must reflect real, unmodified Chopin waltzes.

In [20]:
# Process train and validation subsets
for files_paths, subset_name in [
    (train_files, "train"),
    (val_files, "valid"),
]:
    print(f"\n{'='*60}")
    print(f"Processing {subset_name} set ({len(files_paths)} files)...")
    print(f"{'='*60}")
    
    subset_dir = CHUNKED_DIR / subset_name
    subset_dir.mkdir(parents=True, exist_ok=True)
    
    # ── Step 1: Chunk MIDIs into ~1024 token segments ──
    print(f"\n  Step 1: Chunking into ≤1024-token segments (2-bar overlap)...")
    split_files_for_training(
        files_paths=files_paths,
        tokenizer=tokenizer,
        save_dir=subset_dir,
        max_seq_len=1024,
        num_overlap_bars=2,   # 2-bar overlap between chunks
    )
    
    chunks_after_split = list(subset_dir.glob("**/*.mid"))
    print(f"  Created {len(chunks_after_split)} chunks")
    
    # ── Step 2: Augment (training set ONLY) ──
    if subset_name == "train":
        print(f"\n  Step 2: Augmenting (pitch ±12, velocity ±4, duration ±0.5)...")
        augment_dataset(
            subset_dir,
            pitch_offsets=[-12, 12],
            velocity_offsets=[-4, 4],
            duration_offsets=[-0.5, 0.5],
        )
        chunks_after_aug = list(subset_dir.glob("**/*.mid"))
        print(f"  After augmentation: {len(chunks_after_aug)} total chunks")
        print(f"  Augmentation multiplier: {len(chunks_after_aug) / len(chunks_after_split):.1f}x")
    else:
        print(f"\n  Step 2: Skipping augmentation (validation set)")
        chunks_after_aug = chunks_after_split
    
    print(f"\n  ✓ {subset_name} set ready: {len(chunks_after_aug)} MIDI chunks in {subset_dir}")


Processing train set (16 files)...

  Step 1: Chunking into ≤1024-token segments (2-bar overlap)...


Splitting music files (/Users/jawadmallat/Desktop/Chopin_Waltz_Completion/data/chunked/train): 100%|██████████| 16/16 [00:00<00:00, 418.45it/s]


  Created 97 chunks

  Step 2: Augmenting (pitch ±12, velocity ±4, duration ±0.5)...


Performing data augmentation: 100%|██████████| 97/97 [00:00<00:00, 669.63it/s]


  After augmentation: 643 total chunks
  Augmentation multiplier: 6.6x

  ✓ train set ready: 643 MIDI chunks in /Users/jawadmallat/Desktop/Chopin_Waltz_Completion/data/chunked/train

Processing valid set (3 files)...

  Step 1: Chunking into ≤1024-token segments (2-bar overlap)...


Splitting music files (/Users/jawadmallat/Desktop/Chopin_Waltz_Completion/data/chunked/valid): 100%|██████████| 3/3 [00:00<00:00, 760.66it/s]

  Created 6 chunks

  Step 2: Skipping augmentation (validation set)

  ✓ valid set ready: 6 MIDI chunks in /Users/jawadmallat/Desktop/Chopin_Waltz_Completion/data/chunked/valid


## 2.5 — Prepare the Lost Waltz Seed

The lost waltz is tokenized separately. In Phase 4 (generation), we will  
feed its token sequence as a **prompt** to the fine-tuned model, which will  
then generate a continuation in Chopin's style.

In [21]:
# Tokenize the lost waltz and save its token IDs
lost_score = Score(str(lost_waltz_path))
lost_tokens = tokenizer(lost_score)

# Save the token IDs as JSON for easy loading in Phase 4
seed_data = {
    "source": "New_Chopin_Waltz_.mid",
    "description": "Lost Chopin Waltz in A minor (c. 1830-1835), Morgan Library",
    "num_tracks": len(lost_tokens),
    "tracks": []
}

for i, seq in enumerate(lost_tokens):
    seed_data["tracks"].append({
        "track_index": i,
        "num_tokens": len(seq.ids),
        "token_ids": seq.ids,
    })
    print(f"Track {i}: {len(seq.ids)} tokens")

seed_json_path = SEED_DIR / "lost_waltz_tokens.json"
with open(seed_json_path, "w") as f:
    json.dump(seed_data, f, indent=2)

print(f"\nSeed tokens saved to {seed_json_path}")
print(f"Total tokens across all tracks: {sum(len(s.ids) for s in lost_tokens)}")
print(f"\nThis will be the prompt for generation in Phase 4.")

Track 0: 656 tokens
Track 1: 504 tokens

Seed tokens saved to /Users/jawadmallat/Desktop/Chopin_Waltz_Completion/data/seed/lost_waltz_tokens.json
Total tokens across all tracks: 1160

This will be the prompt for generation in Phase 4.


## 2.6 — Verification & Statistics

Let's verify everything looks right before moving to Phase 3.

In [22]:
import numpy as np

print("=" * 60)
print("PHASE 2 — FINAL VERIFICATION")
print("=" * 60)

# Count files
train_chunks = list((CHUNKED_DIR / "train").glob("**/*.mid"))
valid_chunks = list((CHUNKED_DIR / "valid").glob("**/*.mid"))

print(f"\n--- File Counts ---")
print(f"Training chunks:   {len(train_chunks)}")
print(f"Validation chunks: {len(valid_chunks)}")
print(f"Lost waltz seed:   1 file ({seed_json_path.name})")

# Tokenize a sample of training chunks to check lengths
print(f"\n--- Token Length Verification (sampling 20 training chunks) ---")
sample_chunks = train_chunks[:20] if len(train_chunks) >= 20 else train_chunks
token_lengths = []

for chunk_path in sample_chunks:
    try:
        score = Score(str(chunk_path))
        tokens = tokenizer(score)
        for seq in tokens:
            token_lengths.append(len(seq.ids))
    except Exception as e:
        print(f"  WARNING: Failed to tokenize {chunk_path.name}: {e}")

if token_lengths:
    lengths_arr = np.array(token_lengths)
    print(f"  Sampled {len(token_lengths)} sequences")
    print(f"  Token lengths: mean={lengths_arr.mean():.0f}, "
          f"min={lengths_arr.min()}, max={lengths_arr.max()}, "
          f"std={lengths_arr.std():.0f}")
    over_1024 = (lengths_arr > 1024).sum()
    if over_1024 > 0:
        print(f"  ⚠️  {over_1024} sequences exceed 1024 tokens (will be truncated during training)")
    else:
        print(f"  ✓ All sampled sequences are ≤1024 tokens")

# Roundtrip test on a training chunk
print(f"\n--- Roundtrip Test ---")
test_chunk = train_chunks[0]
test_score = Score(str(test_chunk))
test_tokens = tokenizer(test_score)
test_roundtrip = tokenizer(test_tokens)
roundtrip_out = ANALYSIS_DIR / "roundtrip_test_chunk.mid"
test_roundtrip.dump_midi(str(roundtrip_out))
print(f"  Original chunk: {test_chunk.name}")
print(f"  Roundtrip saved: {roundtrip_out}")
print(f"  → Open both in MuseScore to verify they sound the same")

# Vocabulary check
print(f"\n--- Vocabulary ---")
print(f"  Tokenizer vocab size: {len(tokenizer)}")
print(f"  Expected: ~20,000 (BPE)")
print(f"  Match: {'✓' if len(tokenizer) > 15000 else '⚠️ CHECK THIS'}")

PHASE 2 — FINAL VERIFICATION

--- File Counts ---
Training chunks:   643
Validation chunks: 6
Lost waltz seed:   1 file (lost_waltz_tokens.json)

--- Token Length Verification (sampling 20 training chunks) ---
  Sampled 20 sequences
  Token lengths: mean=638, min=192, max=808, std=147
  ✓ All sampled sequences are ≤1024 tokens

--- Roundtrip Test ---
  Original chunk: Waltz_3_op34no2-in-A-minor_t0_0#v-4.mid
  Roundtrip saved: /Users/jawadmallat/Desktop/Chopin_Waltz_Completion/analysis/roundtrip_test_chunk.mid
  → Open both in MuseScore to verify they sound the same

--- Vocabulary ---
  Tokenizer vocab size: 20000
  Expected: ~20,000 (BPE)
  Match: ✓


In [23]:
# Save Phase 2 summary for documentation
summary = {
    "phase": "Phase 2: Tokenization & Data Preparation",
    "tokenizer": {
        "source": "Natooz/Maestro-REMI-bpe20k",
        "type": "REMI + BPE",
        "vocab_size": len(tokenizer),
        "saved_to": str(tokenizer_save_path),
    },
    "data_split": {
        "train_waltzes": [f.name for f in train_files],
        "valid_waltzes": [f.name for f in val_files],
        "lost_waltz": "New_Chopin_Waltz_.mid (excluded from training)",
    },
    "chunking": {
        "max_seq_len": 1024,
        "num_overlap_bars": 2,
    },
    "augmentation": {
        "applied_to": "training set only",
        "pitch_offsets": [-12, 12],
        "velocity_offsets": [-4, 4],
        "duration_offsets": [-0.5, 0.5],
    },
    "output": {
        "train_chunks": len(train_chunks),
        "valid_chunks": len(valid_chunks),
        "seed_file": str(seed_json_path),
    },
}

summary_path = ANALYSIS_DIR / "phase2_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Phase 2 summary saved to {summary_path}")
print(f"\n{'='*60}")
print(f"PHASE 2 COMPLETE")
print(f"{'='*60}")
print(f"\nYour data is ready for Phase 3 (fine-tuning).")
print(f"\nNext steps:")
print(f"  1. Commit: git add . && git commit -m 'Phase 2: tokenization complete'")
print(f"  2. Upload data/chunked/ folder to Google Drive for Colab access")
print(f"  3. Proceed to Phase 3 notebook (runs on Colab with GPU)")

Phase 2 summary saved to /Users/jawadmallat/Desktop/Chopin_Waltz_Completion/analysis/phase2_summary.json

PHASE 2 COMPLETE

Your data is ready for Phase 3 (fine-tuning).

Next steps:
  1. Commit: git add . && git commit -m 'Phase 2: tokenization complete'
  2. Upload data/chunked/ folder to Google Drive for Colab access
  3. Proceed to Phase 3 notebook (runs on Colab with GPU)


---

## Summary & Notes

**What we did:**
- Downloaded the pre-trained REMI+BPE tokenizer (20k vocabulary) from the Maestro model
- Split 19 waltzes into 16 train + 3 validation
- Chunked all waltzes into ≤1024-token segments with 2-bar overlap
- Augmented training chunks (pitch ±12 semitones, velocity ±4, duration ±0.5)
- Tokenized the lost waltz separately as the generation seed

**Why we used the pre-trained tokenizer (not a custom one):**
The Maestro model's 50M+ parameters were trained to predict the next token using this exact vocabulary. Changing the vocabulary would require training from scratch on only 19 pieces — catastrophic overfitting. The pre-trained tokenizer is suboptimal for Chopin specifically (sequences ~15-20% longer than a Chopin-specific BPE would produce), but the trade-off is worth it.

**Why we only augment training, not validation:**
Validation measures how well the model generalizes to *real* Chopin waltzes, not augmented ones.

**Next:** Phase 3 — Fine-tuning the pre-trained model on Google Colab.